# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder: OPTIONAL to execute C++ code</h2>
            <span style="color:#f71;">As an alternative, you can run it on the website given yesterday</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important Note</h1>
            <span style="color:#900;">
            In this lab, I use free open source models on Ollama. I also use paid open-source models via Groq and OpenRouter. Only pick the models you want to!
            </span>
        </td>
    </tr>
</table>

In [1]:
# imports

import os
import io
import sys
from openai import OpenAI
import gradio as gr
import subprocess


In [2]:
# Connect to client libraries
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=OPENROUTER_API_KEY)

OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(api_key="ollama", base_url=OLLAMA_BASE_URL)

OPENROUTER_FREE = "openrouter/free"
OPENROUTER_OPENAI_MODEL = "openai/gpt-5.6-sol"
OPENROUTER_GROK_MODEL = "x-ai/grok-4.5"
OPENROUTER_GPT_OSS_MODEL = "openai/gpt-oss-120b"
OPENROUTER_XIAOMI_MODEL = "xiaomi/mimo-v2.5"
OPENROUTER_VARIOUS = "moonshotai/kimi-k3"

OLLAMA_GEMMA4 = "gemma4:latest"
OLLAMA_GPT_OSS = "gpt-oss:20b"
OLLAMA_QWEN = "qwen2.5-coder:14b"
OLLAMA_IBM = "ibm/granite4.1:8b"


In [3]:
models = [OPENROUTER_VARIOUS, OPENROUTER_FREE, OPENROUTER_OPENAI_MODEL, OPENROUTER_GROK_MODEL, OPENROUTER_GPT_OSS_MODEL, OPENROUTER_XIAOMI_MODEL, OLLAMA_GEMMA4, OLLAMA_GPT_OSS, OLLAMA_QWEN, OLLAMA_IBM]

clients = { OPENROUTER_VARIOUS:  openrouter, OPENROUTER_FREE: openrouter, OPENROUTER_OPENAI_MODEL: openrouter, OPENROUTER_GROK_MODEL: openrouter, OPENROUTER_GPT_OSS_MODEL: openrouter, OPENROUTER_XIAOMI_MODEL: openrouter, 
OLLAMA_GEMMA4: ollama, OLLAMA_GPT_OSS: ollama, OLLAMA_QWEN: ollama, OLLAMA_IBM: ollama}



In [4]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Linux',
  'arch': 'x86_64',
  'release': '7.1.4-200.nobara.fc44.x86_64',
  'version': '#1 SMP PREEMPT_DYNAMIC Mon Jul 20 03:39:43 UTC 2026',
  'kernel': '7.1.4-200.nobara.fc44.x86_64',
  'distro': {'name': 'Nobara Linux 44 (KDE Plasma Desktop Edition)',
   'version': '44'},
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'x86_64-redhat-linux'},
 'package_managers': ['dnf', 'yum'],
 'cpu': {'brand': '12th Gen Intel(R) Core(TM) i7-12700',
  'cores_logical': 12,
  'cores_physical': 12,
  'simd': ['AVX', 'AVX2', 'FMA', 'SSE4_2']},
 'toolchain': {'compilers': {'gcc': 'gcc (GCC) 16.1.1 20260515 (Red Hat 16.1.1-2)',
   'g++': 'g++ (GCC) 16.1.1 20260515 (Red Hat 16.1.1-2)',
   'clang': '',
   'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': 'GNU Make 4.4.1'},
  'linkers': {'ld_lld': ''}}}

## Overwrite this with the commands from yesterday

Or just use the website like yesterday:

 https://www.programiz.com/cpp-programming/online-compiler/

In [5]:
compile_command = [
    "g++",                 # the compiler driver
    "main.cpp",            # source file
    "-std=c++20",          # pick a modern C++ standard (optional)
    "-O3",                 # highest conventional optimisation level
    "-march=native",       # enable CPU‑specific instructions (AVX2, etc.)
    "-mtune=native",
    "-flto",               # link‑time optimisation – reduces binary size + speeds up hot paths
    "-funroll-loops",      # allow the compiler to unroll loops for speed
    "-ffast-math",         # relax IEEE strictness (fastest floating‑point)
    "-pipe",               # use pipes instead of temporary files during compilation
    "-s",                  # strip debug symbols – makes binary smaller
    "-o", "main"        # output executable name
]
run_command = ["./main"]


## And now, on with the main task

In [6]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code. Only write the final version of the code. Do not include any reasoning or explanations.
Python code to port:

```python
{python}
```
"""

In [7]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [8]:
def write_output(cpp):
    with open("main.cpp", "w") as f:
        f.write(cpp)

In [9]:
def port(model, python):
    client = clients[model]
    response = client.chat.completions.create(model=model, messages=messages_for(python))
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)
    return reply

In [10]:
pi = """
import time
import gmpy2
from gmpy2 import mpz

def pi_chudnovsky_bs(digits):
    # Each term in the Chudnovsky series yields ~14.1816474627254776 digits
    DIGITS_PER_TERM = 14.1816474627254776
    N = int(digits / DIGITS_PER_TERM) + 1

    # Chudnovsky Constants
    C = 640320
    C3_OVER_24 = C**3 // 24

    def bs(a, b):
        if b - a == 1:
            if a == 0:
                P = mpz(1)
                Q = mpz(1)
                T = mpz(13591409)
            else:
                P = mpz((6 * a - 5) * (2 * a - 1) * (6 * a - 1))
                Q = mpz(a**3 * C3_OVER_24)
                T = P * (13591409 + 545140134 * a)
                if a & 1:
                    T = -T
            return P, Q, T

        # Midpoint for divide-and-conquer binary split
        m = (a + b) // 2
        P_a, Q_a, T_a = bs(a, m)
        P_b, Q_b, T_b = bs(m, b)

        P = P_a * P_b
        Q = Q_a * Q_b
        T = Q_b * T_a + P_a * T_b
        return P, Q, T

    print(f"Starting calculation of Pi to {digits:,} decimal places...")
    start_time = time.time()

    # Step 1: Run binary splitting (pure integer math)
    print("1/3: Computing integer series sum via Binary Splitting...")
    P, Q, T = bs(0, N)

    # Step 2: Final floating point square root and division
    print("2/3: Computing final floating-point operations...")
    bits = int(digits * 3.321928094887362) + 100
    gmpy2.get_context().precision = bits

    # Formula: pi = (Q * 426880 * sqrt(10005)) / T
    numerator = Q * 426880 * gmpy2.sqrt(gmpy2.mpfr(10005))
    pi = numerator / T

    calc_time = time.time() - start_time
    print(f"Math finished in {calc_time:.2f} seconds!")

    # Step 3: Format string conversion
    print("3/3: Formatting integer string representation...")
    fmt_start = time.time()
    pi_str = f"{pi:.{digits}f}"
    fmt_time = time.time() - fmt_start

    return pi_str, calc_time, fmt_time

if __name__ == "__main__":
    target_digits = 10_000_000
    
    pi_str, cpu_time, fmt_time = pi_chudnovsky_bs(target_digits)

    print(f"\n--- Benchmark Summary ---")
    print(f"Calculation time: {cpu_time:.2f}s")
    print(f"String format time: {fmt_time:.2f}s")
    print(f"Total time:       {cpu_time + fmt_time:.2f}s")
    print(f"First 50 digits:  {pi_str[:52]}")
    print(f"Last 10 digits:   {pi_str[-10:]}")
"""

In [11]:
def compile_and_run():
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    except subprocess.CalledProcessError as e:
        print(f"An error occurred:\n{e.stderr}")

In [12]:
import contextlib

# Wrapper that captures stdout
def run_and_capture():
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        compile_and_run()
    return buffer.getvalue()

In [13]:
with gr.Blocks() as ui:
    with gr.Row():
        python = gr.Textbox(label="Python code:", lines=28, value=pi)
        cpp = gr.Textbox(label="C++ code:", lines=28)
    with gr.Row():
        model = gr.Dropdown(models, label="Select model", value=models[0])
        convert = gr.Button("Convert code")
    with gr.Row():
        run_btn = gr.Button("Compile & Run")
        output_box = gr.Textbox(
            label="Output",
            lines=15,
            interactive=False
        )

    convert.click(port, inputs=[model, python], outputs=[cpp])

    # Connect button to function
    run_btn.click(fn=run_and_capture, inputs=[], outputs=output_box)

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [16]:
compile_and_run()